# Exploring LoRA in Fine-Tuning Math Reasoning Models

## Mini Research Problem Notebook

This notebook studies whether compact LoRA-based fine-tuning can improve exact-answer mathematical reasoning under local notebook constraints.

### Project focus
This mini research problem has two parts:

1. **Main experiment (Path A):** a local LoRA rank-sweep on `Qwen/Qwen2.5-Math-1.5B-Instruct` using MLX on Apple silicon.
2. **Feasibility extension (Path B):** a small local LoRA-XS proof-of-concept.

### Main claim developed in this notebook
Under a fixed local training budget, LoRA improves over the base model on held-out exact-answer math benchmarks, and smaller ranks (`r2`, `r4`) outperform the larger `r8` configuration in this setup.

### Scope note
TinyLoRA is discussed as **related work / future work**, not as a fully reproduced benchmark in this notebook.


### Motivation
Inspired by the paper **“Learning to Reason in 13 Parameters,”** this project asks how far ultra-compact parameter-efficient tuning can go in a **local environment**. It also tests whether gains from local math reasoning fine-tuning transfer beyond simple math tasks to **olympiad-style** and **AIMO-style** questions, since standard math benchmarks may not fully reflect broader reasoning ability.


## 0.1 Notebook Organization

The notebook follows a mini research problem structure:

1. **Research question and hypotheses**
2. **Dataset design and benchmark split**
3. **Prompting and answer extraction**
4. **Main method: LoRA rank sweep**

In [1]:
import os
import re
import json
import time
import random
from typing import Dict, List, Any

import pandas as pd

SEED = 42
random.seed(SEED)

print("Environment ready")

Environment ready


## 1. Research Question and Hypotheses

### Research question
Can compact LoRA fine-tuning improve exact-answer math reasoning performance in a local notebook setting, and which adapter rank gives the best trade-off between accuracy and parameter efficiency?

### Hypotheses
- **H1:** LoRA fine-tuning will outperform the untuned base model on held-out math benchmarks.
- **H2:** Smaller ranks may be sufficient for this task because the dataset and training budget are relatively small.
- **H3:** The best raw-accuracy model and the best parameter-efficiency model may not be the same.

### Final emphasis of the project
The main quantitative result comes from a **local LoRA rank sweep** on `Qwen/Qwen2.5-Math-1.5B-Instruct`, followed by held-out evaluation.


## 2. Dataset Design and Benchmark Split

This project uses different datasets for different roles.

### Training and validation data
- **Train:** `data/train_easy_math_120.jsonl`
- **Validation:** `data/val_easy_math_30.jsonl`

These support supervised fine-tuning and training monitoring.

### Development benchmark
- **`benchmark_50`**
  - used during debugging and model iteration
  - useful for fast comparisons
  - should be treated as a **development benchmark**, not a pristine final test

### Held-out benchmarks for final reporting
- **Reasoning-like held-out set:** `data/reasoning_like_benchmark50_100.jsonl`
- **Olympiad-style held-out set:** `data/olympiad_style_100.jsonl`
- **AIMO-style held-out set:** `data/aimo_specific_100.jsonl`

### Experimental logic
- `train.jsonl` updates model weights
- `valid.jsonl` monitors validation loss during training
- held-out benchmarks are used **after training** to measure generalization


In [2]:
TRAIN_PATH = "data/train_easy_math_120.jsonl"
VAL_PATH = "data/val_easy_math_30.jsonl"

REASONING_HELDOUT_PATH = "data/benchmark_50.jsonl"
OLYMPIAD_HELDOUT_PATH = "data/olympiad_style_50.jsonl"
AIMO_HELDOUT_PATH = "data/aimo_specific_100.jsonl"

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)


# load benchmark test data
benchmark_50 = load_jsonl(REASONING_HELDOUT_PATH)
olym_benchmark_50 = load_jsonl(OLYMPIAD_HELDOUT_PATH)
aimo_benchmark_100 = load_jsonl(AIMO_HELDOUT_PATH) 

print(len(train_data), len(val_data))
print(train_data[0])

120 30
{'id': 'train_001', 'type': 'train', 'topic': 'shopping_counting', 'question': 'Maya bought 11 apples and 5 oranges. She used 6 apples to bake a pie and then bought 4 more oranges. How many pieces of fruit does she have now?', 'answer': '14'}


## 3. Prompt Format

All conditions use the same prompt format.

This keeps the comparison fair across:
- base model
- LoRA-r8
- LoRA-r4
- LoRA-r2

The goal is to encourage short exact-answer outputs while avoiding over-complicated prompting.


In [3]:
def make_prompt(question: str) -> str:
    return f'''You are a math assistant.
Return only the final numeric answer.
Do not explain.

Examples:
Question: 12 * 13
Answer: 156

Question: 25 + 17
Answer: 42

Question: 84 / 6 + 15
Answer: 29

Now answer:
Question: {question}
Answer:'''

## 4. Answer Extraction and Normalization

The project is evaluated using **exact extracted final answers**.

Because math-oriented language models may still produce explanations or extra text, the notebook uses a custom extraction function to:
- capture `\boxed{}` answers when present
- recover clean numeric outputs
- ignore obvious explanatory openings
- reduce false positives from echoed prompt text


In [14]:
def extract_final_answer(text: str) -> str:
    text = text.strip()
    if not text:
        return ""

    first_line = text.splitlines()[0].strip()

    bad_starts = (
        "to solve",
        "to find",
        "we need",
        "the expression",
        "the problem",
        "here are",
    )
    if first_line.lower().startswith(bad_starts):
        return ""

    m = re.search(r"\\boxed\{(-?\d+(?:\.\d+)?)\}", first_line)
    if m:
        return m.group(1)

    m = re.search(r"=\s*(-?\d+(?:\.\d+)?)", first_line)
    if m:
        return m.group(1)

    m = re.search(r"(?:final answer is|answer is|answer:)\s*(-?\d+(?:\.\d+)?)", first_line, flags=re.I)
    if m:
        return m.group(1)

    m = re.fullmatch(r"-?\d+(?:\.\d+)?", first_line)
    if m:
        return m.group(0)

    nums = re.findall(r"-?\d+(?:\.\d+)?", first_line)
    return nums[-1] if nums else ""

def normalize_answer(ans: str) -> str:
    return ans.strip().replace(",", "")

def is_correct(pred: str, gold: str) -> bool:
    return normalize_answer(pred) == normalize_answer(gold)

## 5. Main Method: Local MLX / MLX-LM Inference and LoRA Evaluation

The main experiment uses **MLX / MLX-LM** on Apple silicon.

### Base model
- `Qwen/Qwen2.5-Math-1.5B`

### Main adapter conditions
- `lora_r8_250`
- `lora_r4_250`
- `lora_r2_250`

### Why this became the main path
This path produced the stable local results of the project:
- practical training on-device
- reproducible rank-sweep comparisons
- clean held-out benchmarking


In [4]:
from mlx_lm import load, generate

BASE_MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"

model, tokenizer = load(BASE_MODEL_NAME)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

In [6]:
# def run_model(question: str) -> str:
#     prompt = make_prompt(question)
#     response = generate(model, tokenizer, prompt=prompt, max_tokens=20)
#     return response.splitlines()[0].replace("<END>", "").strip()

# def run_model_lora_r8_250(question: str) -> str:
#     prompt = make_prompt(question)
#     response = generate(lora_r8_250_model, lora_r8_250_tokenizer, prompt=prompt, max_tokens=20)
#     return response.splitlines()[0].replace("<END>", "").strip()

# def run_model_lora_r4_250(question: str) -> str:
#     prompt = make_prompt(question)
#     response = generate(lora_r4_250_model, lora_r4_250_tokenizer, prompt=prompt, max_tokens=20)
#     return response.splitlines()[0].replace("<END>", "").strip()

# def run_model_lora_r2_250(question: str) -> str:
#     prompt = make_prompt(question)
#     response = generate(lora_r2_250_model, lora_r2_250_tokenizer, prompt=prompt, max_tokens=20)
#     return response.splitlines()[0].replace("<END>", "").strip()

## 6. Unified Evaluation Helpers

The evaluation code below supports the main metrics used in this mini research problem:

- **final answer accuracy**
- **average output length**
- **consistency across 3 repeated runs**
- **training time**
- **trainable parameter count**
- **accuracy gain per parameter**

A 3-run majority vote is used to reduce noise in the final extracted-answer accuracy.


In [5]:
def evaluate_dataset_multi_run(data, runner, n_runs=3):
    rows = []

    for ex in data:
        outputs = []
        preds = []
        lengths = []

        for _ in range(n_runs):
            raw = runner(ex["question"])
            if raw is None:
                raw = ""
            pred = extract_final_answer(raw)

            outputs.append(raw)
            preds.append(pred)
            lengths.append(len(raw.split()))

        majority_pred = max(set(preds), key=preds.count) if preds else ""
        majority_correct = majority_pred == ex["answer"]
        consistency = len(set(preds)) == 1

        rows.append({
            "id": ex["id"],
            "difficulty": ex.get("difficulty", "unknown"),
            "topic": ex.get("topic", "unknown"),
            "question": ex["question"],
            "gold": ex["answer"],
            "raw_output_1": outputs[0] if len(outputs) > 0 else "",
            "raw_output_2": outputs[1] if len(outputs) > 1 else "",
            "raw_output_3": outputs[2] if len(outputs) > 2 else "",
            "pred_1": preds[0] if len(preds) > 0 else "",
            "pred_2": preds[1] if len(preds) > 1 else "",
            "pred_3": preds[2] if len(preds) > 2 else "",
            "correct_1": preds[0] == ex["answer"] if len(preds) > 0 else False,
            "correct_2": preds[1] == ex["answer"] if len(preds) > 1 else False,
            "correct_3": preds[2] == ex["answer"] if len(preds) > 2 else False,
            "majority_pred": majority_pred,
            "majority_correct": majority_correct,
            "approx_token_len_1": lengths[0] if len(lengths) > 0 else 0,
            "approx_token_len_2": lengths[1] if len(lengths) > 1 else 0,
            "approx_token_len_3": lengths[2] if len(lengths) > 2 else 0,
            "avg_approx_token_len": sum(lengths) / len(lengths) if lengths else 0,
            "consistent_across_3_runs": consistency,
        })

    return pd.DataFrame(rows)

def summarize_metrics(
    df,
    condition_name,
    training_time_sec=0,
    trainable_params=0,
    total_params=None,
    trainable_percent=None,
    base_accuracy=None
):
    final_answer_accuracy = df["majority_correct"].mean()
    avg_token_length = df["avg_approx_token_len"].mean()
    consistency_rate = df["consistent_across_3_runs"].mean()

    accuracy_gain = None
    improvement_per_parameter = None
    accuracy_gain_per_1k_params = None
    accuracy_gain_per_1m_params = None

    if base_accuracy is not None and trainable_params not in (None, 0):
        accuracy_gain = final_answer_accuracy - base_accuracy
        improvement_per_parameter = accuracy_gain / trainable_params
        accuracy_gain_per_1k_params = accuracy_gain / (trainable_params / 1000.0)
        accuracy_gain_per_1m_params = accuracy_gain / (trainable_params / 1_000_000.0)

    return {
        "condition": condition_name,
        "final_answer_accuracy": final_answer_accuracy,
        "avg_token_length": avg_token_length,
        "consistency_across_3_runs": consistency_rate,
        "training_time_sec": training_time_sec,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_percent": trainable_percent,
        "accuracy_gain": accuracy_gain,
        "improvement_per_parameter": improvement_per_parameter,
        "accuracy_gain_per_1k_params": accuracy_gain_per_1k_params,
        "accuracy_gain_per_1m_params": accuracy_gain_per_1m_params,
    }

def show_wrong_details(df, max_rows=10):
    wrong = df[~df["majority_correct"]].head(max_rows)
    print(f"Number of incorrect examples: {len(wrong)}")
    for _, row in wrong.iterrows():
        print("=" * 100)
        print(f"ID: {row['id']}")
        print(f"Difficulty: {row['difficulty']}")
        print(f"Topic: {row.get('topic', 'unknown')}")
        print(f"Question: {row['question']}")
        print(f"Run 1 response: {row['raw_output_1']}")
        print(f"Run 2 response: {row['raw_output_2']}")
        print(f"Run 3 response: {row['raw_output_3']}")
        print(f"Majority extracted answer: {row['majority_pred']}")
        print(f"Label answer: {row['gold']}")

## 7. Condition A: Base Model

The base model serves as the no-adaptation reference point.

Its role is to answer:
- how strong is the untuned local math model?
- how much improvement comes from compact adaptation?


In [ ]:
# held-out evaluation for the base model:
df_reason_base = evaluate_dataset_multi_run(benchmark_50, run_model, n_runs=3)
df_oly_base = evaluate_dataset_multi_run(olym_benchmark_50, run_model, n_runs=3)
df_aimo_base = evaluate_dataset_multi_run(aimo_benchmark_100, run_model, n_runs=3)

## 8. Condition B: LoRA Rank Sweep

This is the core experiment of the notebook.

### Adapter conditions
- `r8`
- `r4`
- `r2`

### Final comparison budget
The strongest result in this notebook comes from the **250-iteration comparison**, which gives all ranks the same longer training budget.

### Research purpose
The rank sweep is intended to reveal:
- whether larger ranks help in this local-data regime
- whether smaller ranks preserve most of the gain
- which rank is best for raw accuracy vs efficiency


In [6]:
configs = {
    "config_lora_r8_300.yaml": """model: Qwen/Qwen2.5-Math-1.5B
train: true
data: data/lora_easy_math_v3
train_type: lora
train_mode: sft
batch_size: 4
learning_rate: 1e-5
iters: 300
adapter_path: adapters/qwen_math_lora_r8_300
lora_parameters:
  rank: 8
  dropout: 0.0
  scale: 10.0
""",
    "config_lora_r4_300.yaml": """model: Qwen/Qwen2.5-Math-1.5B
train: true
data: data/lora_easy_math_v3
train_type: lora
train_mode: sft
batch_size: 4
learning_rate: 1e-5
iters: 300
adapter_path: adapters/qwen_math_lora_r4_300
lora_parameters:
  rank: 4
  dropout: 0.0
  scale: 10.0
""",
    "config_lora_r2_300.yaml": """model: Qwen/Qwen2.5-Math-1.5B
train: true
data: data/lora_easy_math_v3
train_type: lora
train_mode: sft
batch_size: 4
learning_rate: 1e-5
iters: 300
adapter_path: adapters/qwen_math_lora_r2_300
lora_parameters:
  rank: 2
  dropout: 0.0
  scale: 10.0
"""
}
for name, text in configs.items():
    with open(name, "w", encoding="utf-8") as f:
        f.write(text)

print("Wrote config files:", list(configs.keys()))

Wrote config files: ['config_lora_r8_300.yaml', 'config_lora_r4_300.yaml', 'config_lora_r2_300.yaml']


In [9]:
# Example run for YAML-driven MLX training commands used in Path A:
import sys
start = time.time()
!{sys.executable} -m mlx_lm_lora.train -c config_lora_r8_300.yaml
lora_r8_300_training_time_sec = time.time() - start
print("LoRA-r8-300 training time (sec):", round(lora_r8_300_training_time_sec, 2))

start = time.time()
!{sys.executable} -m mlx_lm_lora.train -c config_lora_r4_300.yaml
lora_r4_300_training_time_sec = time.time() - start
print("LoRA-r4-300 training time (sec):", round(lora_r4_300_training_time_sec, 2))

start = time.time()
!{sys.executable} -m mlx_lm_lora.train -c config_lora_r2_300.yaml
lora_r2_300_training_time_sec = time.time() - start
print("LoRA-r2-300 training time (sec):", round(lora_r2_300_training_time_sec, 2))

Python(31139) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



╔══════════════════════════════════════════════════════════════════════════════════════════╗
║                                                                                          ║
║ ███╗   ███╗██╗     ██╗  ██╗    ██╗     ███╗   ███╗    ██╗      ██████╗ ██████╗  █████╗   ║
║ ████╗ ████║██║     ╚██╗██╔╝    ██║     ████╗ ████║    ██║     ██╔═══██╗██╔══██╗██╔══██╗  ║
║ ██╔████╔██║██║      ╚███╔╝     ██║     ██╔████╔█���║    ██║     ██║   ██║██████╔╝███████║  ║
║ ██║╚██╔╝██║██║      ██╔██╗     ██║     ██║╚██╔╝██║    ██║     ██║   ██║██╔══██╗██╔══██║  ║
║ ██║ ╚═╝ ██║███████╗██╔╝ ██╗    ███████╗██║ ╚═╝ ██║    ███████╗╚██████╔╝██║  ██║██║  ██║  ║
║ ╚═╝     ╚═╝╚══════╝╚═╝  ╚═╝    ╚══════╝╚═╝     ╚═╝    ╚══════╝ ╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═╝  ║
║                                                                                          ║
║ Advanced Fine-Tuning Framework                                                           ║
║ LoRA • (Online-)DPO • XPO • CPO • CPO • ORPO • PPO • GRPO • DrGRP

Python(31280) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



╔══════════════════════════════════════════════════════════════════════════════════════════╗
║                                                                                          ║
║ ███╗   ███╗██╗     ██╗  ██╗    ██╗     ███╗   ███╗    ██╗      ██████╗ ██████╗  █████╗   ║
║ ████╗ ████║██║     ╚██╗██╔╝    ██║     ████╗ ████║    ██║     ██╔═══██╗██╔══██╗██╔══██╗  ║
║ ██╔████╔██║██║      ╚███╔╝     ██║     ██╔████╔██║    ██║     ██║   ██║██████╔╝███████║  ║
║ ██║╚██╔╝██║██║      ██╔██╗     ██║     ██║╚██╔╝██║    ██║     ██║   ██║██╔══██╗██╔══██║  ║
║ ██║ ╚═╝ ██║███████╗██╔╝ ██╗    ███████╗██║ ╚═╝ ██║    ███████╗╚██████╔╝██║  ██║██║  ██║  ║
║ ╚═╝     ╚═╝╚══════╝╚═╝  ╚═╝    ╚══════╝╚═╝     ╚═╝    ╚══════╝ ╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═╝  ║
║                                                                                          ║
║ Advanced Fine-Tuning Framework                                                           ║
║ LoRA • (Online-)DPO • XPO • CPO • CPO • ORPO • PPO • GRPO • DrGRPO 

Python(31489) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



╔══════════════════════════════════════════════════════════════════════════════════════════╗
║                                                                                          ║
║ ███╗   ███╗██╗     ██╗  ██╗    ██╗     ███╗   ███╗    ██╗      ██████╗ ██████╗  █████╗   ║
║ ████╗ ████║██║     ╚██╗██╔╝    ██║     ████╗ ████║    ██║     ██╔═══██╗██╔══██╗██╔══██╗  ║
║ ██╔████╔██║██║      ╚███╔╝     ██║     ██╔████╔█���║    ██║     ██║   ██║██████╔╝███████║  ║
║ ██║╚██╔╝██║██║      ██╔██╗     ██║     ██║╚██╔╝██║    ██║     ██║   ██║██╔══██╗██╔══██║  ║
║ ██║ ╚═╝ ██║███████╗██╔╝ ██╗    ███████╗██║ ╚═╝ ██║    ███████╗╚██████╔╝██║  ██║██║  ██║  ║
║ ╚═╝     ╚═╝╚══════╝╚═╝  ╚═╝    ╚══════╝╚═╝     ╚═╝    ╚══════╝ ╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═╝  ║
║                                                                                          ║
║ Advanced Fine-Tuning Framework                                                           ║
║ LoRA • (Online-)DPO • XPO • CPO • CPO • ORPO • PPO • GRPO • DrGRP

In [8]:
# ---------
# Final 300-iteration LoRA models
# ---------
LORA_R8_300_PATH = "adapters/qwen_math_lora_r8_300"
LORA_R4_300_PATH = "adapters/qwen_math_lora_r4_300"
LORA_R2_300_PATH = "adapters/qwen_math_lora_r2_300"

lora_r8_300_model, lora_r8_300_tokenizer = load(LORA_R8_300_PATH)
lora_r4_300_model, lora_r4_300_tokenizer = load(LORA_R4_300_PATH)
lora_r2_300_model, lora_r2_300_tokenizer = load(LORA_R2_300_PATH)

print("Loaded:")
print("- base model")
print("- LoRA-r8-300")
print("- LoRA-r4-300")
print("- LoRA-r2-300")

The tokenizer you are loading from 'adapters/qwen_math_lora_r8_300' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'adapters/qwen_math_lora_r4_300' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'adapters/qwen_math_lora_r2_300' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag 

Loaded:
- base model
- LoRA-r8-300
- LoRA-r4-300
- LoRA-r2-300


In [9]:
LORA_300_METADATA = {
    "lora_r8_300": {
        "training_time_sec": 262.68,
        "trainable_params": 9_232_000,
        "total_params": 1_543_714_000,
        "trainable_percent": 0.598,
        "adapter_path": "adapters/qwen_math_lora_r8_300",
    },
    "lora_r4_300": {
        "training_time_sec": 264.55,
        "trainable_params": 4_616_000,
        "total_params": 1_543_714_000,
        "trainable_percent": 0.299,
        "adapter_path": "adapters/qwen_math_lora_r4_300",
    },
    "lora_r2_300": {
        "training_time_sec": 432.23,
        "trainable_params": 2_308_000,
        "total_params": 1_543_714_000,
        "trainable_percent": 0.150,
        "adapter_path": "adapters/qwen_math_lora_r2_300",
    },
}


LORA_300_METADATA

{'lora_r8_300': {'training_time_sec': 262.68,
  'trainable_params': 9232000,
  'total_params': 1543714000,
  'trainable_percent': 0.598,
  'adapter_path': 'adapters/qwen_math_lora_r8_300'},
 'lora_r4_300': {'training_time_sec': 264.55,
  'trainable_params': 4616000,
  'total_params': 1543714000,
  'trainable_percent': 0.299,
  'adapter_path': 'adapters/qwen_math_lora_r4_300'},
 'lora_r2_300': {'training_time_sec': 432.23,
  'trainable_params': 2308000,
  'total_params': 1543714000,
  'trainable_percent': 0.15,
  'adapter_path': 'adapters/qwen_math_lora_r2_300'}}